# Smart Streetlight Fault Detection

**Use Case Name:** Smart Streetlight Fault Detection Using Computer Vision  
**Authored by:** Syed Hamiz Hassan, Luke Kankanamge Don , Savith Mudunkotuwa, Josh Wong, Rahul Sheoran  
**Duration:** 90 mins  
**Level:** Intermediate  
**Pre-requisite Skills:** Python, Jupyter Notebook, Computer Vision, Deep Learning Basics, YOLO Object Detection, OpenCV, Data Analysis, Pandas, Matplotlib  

---

# Scenario

As a local council maintenance team, we want an automated system that can detect streetlights and identify whether they are functioning correctly, dim, or completely off using nighttime street imagery, so that faulty streetlights can be identified efficiently without requiring manual inspection.

Traditional manual monitoring of streetlights can be time-consuming, expensive, and difficult to scale across large urban environments. This use case demonstrates how computer vision and deep learning can be used to automate streetlight monitoring through object detection and image analysis techniques.

---

# What this use case will teach you

At the end of this use case you will:

- Understand how object detection models such as YOLO can be used in real-world smart city applications
- Learn how to preprocess low-light images for improved detection performance
- Apply computer vision techniques using OpenCV
- Train and evaluate a YOLO-based streetlight detection model
- Perform image-based brightness analysis to classify streetlights as ON, DIM, or OFF
- Compare detection performance across multiple preprocessing techniques
- Generate visualisations and evaluation summaries using Python data analysis libraries
- Understand how AI solutions can support smart infrastructure maintenance

---

# Introduction

Streetlights are a critical component of urban infrastructure, which improve road safety, visibility, and public security during nighttime conditions. However, monitoring large numbers of streetlights manually is inefficient and costly. Faulty or dim streetlights may remain undetected for long periods especially on routes which are less frequently travelled on, leading to safety concerns and increased maintenance delays.

This use case presents an AI-driven smart streetlight fault detection system using computer vision and deep learning techniques. The solution uses the ExDark low-light image dataset alongside a custom-trained YOLO object detection model to identify streetlights in nighttime scenes. Additional RGB brightness analysis is then applied to classify detected streetlights into operational states such as ON, DIM, or OFF.

Multiple image enhancement approaches including brightness adjustment and CLAHE (Contrast Limited Adaptive Histogram Equalization) are evaluated to improve detection performance under challenging low-light conditions. The final system demonstrates how AI technologies can support smart city infrastructure by automating fault detection and improving maintenance efficiency.

In [ ]:
# Path for Dependency folder, kindly change it to your local path
from pathlib import Path

source_root = Path("/Users/hamizhassan16/Documents/MOP-Code/datascience/usecases/DEPENDENCIES/UC00229_Smart_Streetlight_Fault_Detection")

# Data Preperation

**Cell block changed from Code to markdown inorder to prevent accidental re run as output already run and saved in folder Predicted Streetlights folder in dependencies**

from ultralytics import YOLO
model_path = source_root / "runs" / "detect" / "yolo11n.pt"
model = YOLO(model_path)
model.train(
    data=source_root / "Training Dataset" / "data.yaml",
    epochs=50,
    workers=0
)

In [ ]:
from math import sqrt
def distance(x, y):
    return sqrt(x**2 + y**2)


def centre_distance(box1, box2):
    # box format: [x, y, w, h]
    return distance(box1[0] - box2[0], box1[1] - box2[1])


def size_difference(box1, box2):
    # box format: [x, y, w, h]
    width_diff = abs(box1[2] - box2[2])
    height_diff = abs(box1[3] - box2[3])
    return width_diff, height_diff


def closestcentre(true_box, predicted_boxes):
    min_distance = float("inf")
    closest_index = -1

    for i in range(len(predicted_boxes)):
        d = centre_distance(true_box, predicted_boxes[i])

        if d < min_distance:
            min_distance = d
            closest_index = i

    return closest_index, min_distance


def load_yolo_labels(label_path):
    boxes = []

    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            values = line.strip().split()

            if len(values) == 5:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])
                boxes.append([x, y, w, h, class_id])

    return boxes

In [ ]:
from ultralytics import YOLO
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
val_image_dir = source_root / "Training Dataset" / "valid" / "images"
val_label_dir = source_root / "Training Dataset" / "valid" / "labels"
model_path = source_root / "runs" / "detect" / "yolo11n.pt"
model = YOLO(model_path)

val_rows = []

val_image_paths = (
    list(val_image_dir.glob("*.jpg")) +
    list(val_image_dir.glob("*.jpeg")) +
    list(val_image_dir.glob("*.png"))
)

for val_image_path in val_image_paths:
    val_label_path = val_label_dir / f"{val_image_path.stem}.txt"

    true_boxes = load_yolo_labels(val_label_path)

    results = model(str(val_image_path), verbose=False)

    pred_xywhn = results[0].boxes.xywhn.cpu().numpy()
    pred_classes = results[0].boxes.cls.cpu().numpy()

    pred_boxes = []
    for box, cls in zip(pred_xywhn, pred_classes):
        x, y, w, h = box
        pred_boxes.append([x, y, w, h, int(cls)])

    image_centre_distances = []
    image_width_differences = []
    image_height_differences = []
    class_matches = []

    unmatched_pred_indexes = set(range(len(pred_boxes)))

    for true_box in true_boxes:
        if len(pred_boxes) == 0:
            continue

        true_xywh = true_box[:4]

        available_preds = [pred_boxes[i][:4] for i in unmatched_pred_indexes]

        if len(available_preds) == 0:
            continue

        closest_local_idx, centre_dist = closestcentre(true_xywh, available_preds)

        actual_pred_idx = list(unmatched_pred_indexes)[closest_local_idx]
        matched_pred = pred_boxes[actual_pred_idx]

        unmatched_pred_indexes.remove(actual_pred_idx)

        width_diff, height_diff = size_difference(true_xywh, matched_pred[:4])

        image_centre_distances.append(centre_dist)
        image_width_differences.append(width_diff)
        image_height_differences.append(height_diff)
        class_matches.append(int(true_box[4] == matched_pred[4]))

    val_rows.append({
        "image": val_image_path.name,
        "actual_objects": len(true_boxes),
        "detected_objects": len(pred_boxes),
        "object_count_difference": abs(len(true_boxes) - len(pred_boxes)),
        "matched_objects": len(image_centre_distances),
        "avg_centre_distance": np.mean(image_centre_distances) if image_centre_distances else np.nan,
        "avg_width_difference": np.mean(image_width_differences) if image_width_differences else np.nan,
        "avg_height_difference": np.mean(image_height_differences) if image_height_differences else np.nan,
        "class_accuracy": np.mean(class_matches) if class_matches else np.nan
    })

val_val_metrics_df = pd.DataFrame(val_rows)

val_val_metrics_df

In [ ]:
val_val_metrics_df[val_val_metrics_df["actual_objects"] == val_val_metrics_df["detected_objects"]]

In [ ]:
summary_df = pd.DataFrame({
    "metric": [
        "Images evaluated",
        "Average actual objects",
        "Average detected objects",
        "Average object count difference",
        "Average matched objects",
        "Average centre distance",
        "Average width difference",
        "Average height difference",
        "Average class accuracy"
    ],
    "value": [
        len(val_val_metrics_df),
        val_val_metrics_df["actual_objects"].mean(),
        val_val_metrics_df["detected_objects"].mean(),
        val_val_metrics_df["object_count_difference"].mean(),
        val_val_metrics_df["matched_objects"].mean(),
        val_val_metrics_df["avg_centre_distance"].mean(),
        val_val_metrics_df["avg_width_difference"].mean(),
        val_val_metrics_df["avg_height_difference"].mean(),
        val_val_metrics_df["class_accuracy"].mean()
    ]
})

print(summary_df)

**Changed Cell to Markdown to avoid large runtimes, as output already exists in Predicted Streetlights folder**

# Root folder containing image folders
image_root = source_root / "ExDark"
source_folders = ["Bicycle", "Bus", "Car", "Motorbike"]

# Output root
output_root = source_root / "Predicted Streetlights"

conf_threshold = 0.25

# Supported image types
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# Create output folders
lights_img_dir = output_root / "Lights Detected" / "images"
lights_lbl_dir = output_root / "Lights Detected" / "labels"
nolights_img_dir = output_root / "No lights Detected" / "images"

lights_img_dir.mkdir(parents=True, exist_ok=True)
lights_lbl_dir.mkdir(parents=True, exist_ok=True)
nolights_img_dir.mkdir(parents=True, exist_ok=True)


total_images = 0
images_with_lights = 0
images_without_lights = 0

for folder_name in source_folders:
    folder_path = image_root / folder_name

    if not folder_path.exists():
        print(f"Skipping missing folder: {folder_path}")
        continue

    for image_path in folder_path.iterdir():
        if image_path.suffix.lower() not in img_exts:
            continue

        total_images += 1
        print(f"Processing: {image_path}")

        # Run inference
        results = model(str(image_path), conf=conf_threshold, verbose=False)
        result = results[0]

        # boxes.xywhn gives normalized x_center, y_center, width, height
        boxes_xywhn = result.boxes.xywhn
        boxes_cls = result.boxes.cls

        # Build YOLO-format label lines
        label_lines = []
        for cls_tensor, box_tensor in zip(boxes_cls, boxes_xywhn):
            cls_id = int(cls_tensor.item())
            x_center, y_center, width, height = box_tensor.tolist()

            line = f"{cls_id} {x_center} {y_center} {width} {height}"
            label_lines.append(line)

        # Create unique output image name to avoid collisions across folders
        output_stem = f"{folder_name}_{image_path.stem}"
        output_image_name = output_stem + image_path.suffix
        output_label_name = output_stem + ".txt"

        if len(label_lines) > 0:
            images_with_lights += 1

            # Copy image
            shutil.copy2(image_path, lights_img_dir / output_image_name)

            # Save label txt
            with open(lights_lbl_dir / output_label_name, "w", encoding="utf-8") as f:
                f.write("\n".join(label_lines))

        else:
            images_without_lights += 1

            # Copy image only
            shutil.copy2(image_path, nolights_img_dir / output_image_name)


print("\nDone.")
print(f"Total images processed: {total_images}")
print(f"Images with lights detected: {images_with_lights}")
print(f"Images with no lights detected: {images_without_lights}")

**Output Summary:**

Done.
Total images processed: 2320

Images with lights detected: 1023

Images with no lights detected: 1297

# Computer Vision Model

**Installing YOLO on system**

import subprocess
import sys

def install(package):
    try:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    except subprocess.CalledProcessError:
        print(f"Failed to install {package}")

def upgrade_pip():
    print("Upgrading pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

def main():
    print("Starting installation...")

    # Step 1: Upgrade pip
    upgrade_pip()

    # Step 2: Install core packages
    packages = [
        "ultralytics",
        "opencv-python",
        "numpy",
        "pillow",
        "matplotlib",
        "torch",
        "torchvision"
    ]

    for pkg in packages:
        install(pkg)

    print("\nInstallation complete!")

    # Step 3: Verify installation
    try:
        from ultralytics import YOLO
        print("\nUltralytics imported successfully!")

        model = YOLO("yolo11n.pt")
        print("YOLO11 model loaded successfully!")

    except Exception as e:
        print("Error verifying installation:", e)


if __name__ == "__main__":
    main()

Training Model - Commented this part results for it have been saved in train3/best.pt

**Turned to markdown cell to avoid running as model already trained, this code is for reference**

from ultralytics import YOLO 

model = YOLO("yolo11n.pt")

model.train(
    data="street_light_dataset/data.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    fraction=0.25
)

Setup

In [ ]:
from ultralytics import YOLO
import os
import cv2
import shutil
import csv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Experiments performed

**Code in markdown cell to show process**

model = YOLO("runs/detect/train3/weights/best.pt")
base_path = "ExDark"
target_classes = ["Car", "Bicycle", "Bus", "Motorbike"]
count = 0
max_images = 200
output_directory = "experiment_results"
os.makedirs(output_directory, exist_ok=True)

Preprocessing Functions

In [ ]:
def brighten_image(image, alpha = 1.5, beta = 30):
    return cv2.convertScaleAbs(image, alpha = alpha, beta = beta)

def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    merged = cv2.merge((l_clahe, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def brighten_then_clahe(image):
    bright = brighten_image(image)
    return apply_clahe(bright)

Defining Experiments

experiments = {
    "experiment_1_original": lambda img: img,
    "experiment_2_brightened": lambda img: brighten_image(img),
    "experiment_3_clahe": lambda img: apply_clahe(img),
    "experiment_4_brightened_clahe": lambda img: brighten_then_clahe(img)
}

Selecting Images

selected_images = []
count = 0
for cls in target_classes:
    class_dir = os.path.join(base_path, cls)
    if not os.path.isdir(class_dir):
        print(f"Directory {class_dir} does not exist. Skipping class {cls}.")
        continue
    for img_name in sorted(os.listdir(class_dir)):
        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_dir, img_name)
            selected_images.append((cls, img_name, img_path))
            count += 1
            if count >= max_images:
                break
    if count >= max_images:
        break

print(f"Selected {len(selected_images)} images.")

Running Experiments

summary = {}

for exp_name, preprocess_func in experiments.items():
    print(f"\nRunning {exp_name} ...")
    
    exp_folder = os.path.join(output_directory, exp_name)
    with_dir = os.path.join(exp_folder, "with_streetlight")
    without_dir = os.path.join(exp_folder, "without_streetlight")
    
    os.makedirs(with_dir, exist_ok=True)
    os.makedirs(without_dir, exist_ok=True)
    
    detected_count = 0
    not_detected_count = 0
    
    for cls, img_name, img_path in selected_images:
        img = cv2.imread(img_path)
        
        if img is None:
            print(f"Failed to read image {img_path}. Skipping.")
            continue
        
        processed_img = preprocess_func(img)
        results = model(processed_img, conf = 0.25, verbose = False)
        new_name = f"{cls}_{img_name}"
        
        if len(results[0].boxes)>0:
            detected_count += 1
            shutil.copy(img_path, os.path.join(with_dir, new_name))
        else:
            not_detected_count += 1
            shutil.copy(img_path, os.path.join(without_dir, new_name))

    summary[exp_name] = {"With Streetlight": detected_count, "Without Streetlight": not_detected_count}

Results for experiments

In [ ]:
# Commented These to show execytion structure and output without re-running
# print("\nSummary of Results:")
# for exp_name, results in summary.items():
#    print(f"\n{exp_name}:")
#    print(f"  With Streetlight: {results['With Streetlight']} images")
#    print(f"  Without Streetlight: {results['Without Streetlight']} images")

Choosing experiment 4 to Perform Experiment 5 to double check images marked as without streetlight

source_experiment = "experiment_4_brightened_clahe"
source_without_dir = os.path.join(output_directory, source_experiment, "without_streetlight")

experiment_5_name = "experiment_5_double_check_from_experiment_4"
experiment_5_folder = os.path.join(output_directory, experiment_5_name)

exp5_with_dir = os.path.join(experiment_5_folder, "with_streetlight")
exp5_without_dir = os.path.join(experiment_5_folder, "without_streetlight")

os.makedirs(exp5_with_dir, exist_ok=True)
os.makedirs(exp5_without_dir, exist_ok=True)

def double_check_preprocess(image):
    return brighten_then_clahe(image)

exp5_detected = 0
exp5_not_detected = 0

for img_name in sorted(os.listdir(source_without_dir)):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(source_without_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            print(f"Failed to read {img_path}")
            continue

        processed_img = double_check_preprocess(image)

        # lower confidence for second-pass checking to make it more sensitive
        results = model(processed_img, conf=0.10, verbose=False)

        if len(results[0].boxes) > 0:
            exp5_detected += 1
            shutil.copy(img_path, os.path.join(exp5_with_dir, img_name))
        else:
            exp5_not_detected += 1
            shutil.copy(img_path, os.path.join(exp5_without_dir, img_name))

print("Experiment 5 complete.")
print("With streetlight:", exp5_detected)
print("Without streetlight:", exp5_not_detected)

**Output**

Experiment 5 complete.
With streetlight: 15
Without streetlight: 180


Updated Summary

In [ ]:
# summary[experiment_5_name] = {
#     "With Streetlight": exp5_detected,
#     "Without Streetlight": exp5_not_detected
# }
# print(summary[experiment_5_name])

# print("\n===== UPDATED SUMMARY =====")
# for exp_name, result in summary.items():
#     print(f"\n{exp_name}:")
#     print(f"  With streetlight : {result['With Streetlight']}")
#     print(f"  Without streetlight : {result['Without Streetlight']}")

RGB output setup

exp5_with_dir = os.path.join(
    output_directory,
    "experiment_5_double_check_from_experiment_4",
    "with_streetlight"
)
print("Experiment 5 folder:", exp5_with_dir)

In [ ]:
#import numpy as np
#import cv2

def classify_streetlight_state(crop, off_threshold=80, dim_threshold=160):
    """
    Improved classification using brightest pixels instead of full crop average
    """

    if crop is None or crop.size == 0:
        return "unknown", 0

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    # Flatten and take top 10% brightest pixels
    pixels = gray.flatten()
    pixels.sort()

    top_pixels = pixels[int(0.9 * len(pixels)):]  # top 10%
    mean_brightness = float(np.mean(top_pixels))

    # Classification
    if mean_brightness < off_threshold:
        state = "off"
    elif mean_brightness < dim_threshold:
        state = "dim"
    else:
        state = "on"

    return state, mean_brightness

Analyze detected images

experiment5_detailed_results = []
image_counts_list = []

total_streetlights = 0
total_on = 0
total_dim = 0
total_off = 0

for img_name in sorted(os.listdir(exp5_with_dir)):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(exp5_with_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        # SAME conditions as Experiment 5 to keep consistency
        processed_img = double_check_preprocess(image)
        results = model(processed_img, conf=0.10, verbose=False)

        boxes = results[0].boxes
        num_lights = len(boxes)

        on_count = 0
        dim_count = 0
        off_count = 0

        per_image_states = []

        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(image.shape[1], x2)
            y2 = min(image.shape[0], y2)

            crop = image[y1:y2, x1:x2]

            state, brightness = classify_streetlight_state(crop)

            if state == "on":
                on_count += 1
                total_on += 1
            elif state == "dim":
                dim_count += 1
                total_dim += 1
            elif state == "off":
                off_count += 1
                total_off += 1

            per_image_states.append({
                "bbox": (x1, y1, x2, y2),
                "state": state,
                "brightness": brightness
            })

            total_streetlights += 1

        # store per-image result
        experiment5_detailed_results.append({
            "image_name": img_name,
            "streetlight_count": num_lights,
            "on": on_count,
            "dim": dim_count,
            "off": off_count,
            "details": per_image_states
        })

        image_counts_list.append((img_name, num_lights))

print("Analysis complete.")
print("Total streetlights:", total_streetlights)
print("On:", total_on)
print("Dim:", total_dim)
print("Off:", total_off)

**Output**

Analysis complete.
Total streetlights: 21
On: 6
Dim: 6
Off: 9

Amount of lights on/off/dim

**Training another model, which would be trained on whole of dataset instead of just a fraction of it**

from ultralytics import YOLO

model_2 = YOLO("yolo11n.pt")
model_2.train(
    data="street_light_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    fraction=1.0
)

Changed above cell to markdown as dont want to accidentaly trigger another training however above cell was run as code block to train the newer model with 100% of data

Testing with improved model and comparing

In [ ]:
import os
from ultralytics import YOLO

path_test = source_root / "runs" / "detect" / "train5" / "weights" / "best.pt"
model_test = YOLO(path_test) # model trained with 100% data
path_old = source_root / "runs" / "detect" / "train3" / "weights" / "best.pt"
model_old = YOLO(path_old) # model trained with 25% data
base_path = source_root / "ExDark"
target_classes = ["Car", "Bicycle", "Bus", "Motorbike"]

count = 0
max_images = 250  # limit
images_with_detections_test = 0
images_with_detections_old = 0

for cls in target_classes:
    folder = os.path.join(base_path, cls)

    for img in os.listdir(folder):
        if img.endswith((".jpg", ".png")):
            img_path = os.path.join(folder, img)

            results_test = model_test(img_path, verbose=False)
            if len(results_test[0].boxes) > 0:
                images_with_detections_test += 1

            results_old = model_old(img_path, verbose=False)
            if len(results_old[0].boxes) > 0:
                images_with_detections_old += 1

            count += 1
            if count >= max_images:
                break
    if count >= max_images:
        break

print("Processed images:", count)
print("Images with detections (Test):", images_with_detections_test)
print("Images with detections (Old):", images_with_detections_old)

Difference in performance can be clearly seen, Therefore model with larger dataset would be preferred over the older version. Ps newer model took 450.66 minutes to train. Now we will duplicate some of the experiments performed above with newer model for better accuracy

# Training Model WIth ExDark Dataset

# Using Groupmates Dataset
**Putting Code in markdown to avoid large runtimes as model already trained**

dataset_path = "Savith_data_exdark_train"
data_yaml = os.path.join(dataset_path, "data.yaml")
model = YOLO("yolo11n.pt")
results = model.train(
    data=data_yaml,
    epochs=200,
    imgsz=640,
    batch=4,
    patience=50,
    name="streetlight_savith_best",
    project="runs/detect",

    # augmentation for small dataset
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,

    # optimization
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    # saving best model
    save=True,
    plots=True
)

Validating

best_model_path = source_root / "runs" / "detect" / "streetlight_savith_best" / "weights" / "best.pt"
best_model = YOLO(best_model_path)
best_model.predict(
    source=source_root / "Savith_data_exdark_train" / "valid" / "images",
    conf=0.25,
    save=True
)

# Running on Model

model_path = source_root / "runs" / "detect" / "streetlight_savith_best" / "weights" / "best.pt"
model = YOLO(model_path)

# Input ExDark folder
base_path = source_root / "ExDark"

# Output folder inside experiment_results
experiment_folder = os.path.join("experiment_results", "experiment_6_original_new_model")

with_folder = os.path.join(experiment_folder, "images_with_streetlights")
without_folder = os.path.join(experiment_folder, "images_without_streetlights")

os.makedirs(with_folder, exist_ok=True)
os.makedirs(without_folder, exist_ok=True)

# Target folders
target_classes = ["Car", "Bicycle", "Bus", "MotorBike"]

# Image formats
image_extensions = [".jpg", ".jpeg", ".png"]

total_images = 0
with_count = 0
without_count = 0

for class_name in target_classes:
    class_path = os.path.join(base_path, class_name)

    if not os.path.exists(class_path):
        print(f"Folder not found: {class_path}")
        continue

    for image_name in os.listdir(class_path):
        if not any(image_name.lower().endswith(ext) for ext in image_extensions):
            continue

        image_path = os.path.join(class_path, image_name)
        total_images += 1

        # Run detection
        results = model.predict(
            source=image_path,
            conf=0.25,
            verbose=False
        )

        boxes = results[0].boxes
        save_name = f"{class_name}_{image_name}"

        if boxes is not None and len(boxes) > 0:
            shutil.copy(image_path, os.path.join(with_folder, save_name))
            with_count += 1
        else:
            shutil.copy(image_path, os.path.join(without_folder, save_name))
            without_count += 1

print("\n=== Experiment 6 Completed ===")
print(f"Total images: {total_images}")
print(f"Images with streetlights: {with_count}")
print(f"Images without streetlights: {without_count}")
print(f"Saved in: {experiment_folder}")


**Output Experiment 6**

=== Experiment 6 Completed ===

Total images: 2320

Images with streetlights: 495

Images without streetlights: 1825

Saved in: experiment_results/experiment_6_original_new_model

# Model Finalized (Experiment 6 model)
# ------------------------------------

# Integrate output from Savith, Double check images and Running RGB Analysis

In [ ]:
savith_images_dir = source_root / "Predicted Streetlights" / "Lights Detected" / "images"

model_path = source_root / "runs" / "detect" / "streetlight_savith_best" / "weights" / "best.pt"
model = YOLO(model_path)
# Results storage
savith_detailed_results = []
savith_image_counts_list = []

total_streetlights = 0
total_on = 0
total_dim = 0
total_off = 0

# Loop through Savith's detected images
for img_path in sorted(savith_images_dir.iterdir()):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    image = cv2.imread(str(img_path))
    if image is None:
        print(f"Could not read image: {img_path.name}")
        continue

    # Validate using trained YOLO model
    results = model.predict(
        source=str(img_path),
        conf=0.10,
        verbose=False
    )

    boxes = results[0].boxes
    num_lights = len(boxes)

    on_count = 0
    dim_count = 0
    off_count = 0

    per_image_states = []

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        # keep coordinates inside image
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(image.shape[1], x2)
        y2 = min(image.shape[0], y2)

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        # RGB / brightness classification
        state, brightness = classify_streetlight_state(crop)

        if state == "on":
            on_count += 1
            total_on += 1
        elif state == "dim":
            dim_count += 1
            total_dim += 1
        elif state == "off":
            off_count += 1
            total_off += 1

        per_image_states.append({
            "bbox": (x1, y1, x2, y2),
            "state": state,
            "brightness": brightness
        })

        total_streetlights += 1

    savith_detailed_results.append({
        "image_name": img_path.name,
        "streetlight_count": num_lights,
        "on": on_count,
        "dim": dim_count,
        "off": off_count,
        "details": per_image_states
    })

    savith_image_counts_list.append((img_path.name, num_lights))

print("Savith image validation + RGB analysis complete.")
print("Total streetlights:", total_streetlights)
print("On:", total_on)
print("Dim:", total_dim)
print("Off:", total_off)

**Saving results of original dataset rgb analysis**

In [ ]:
output_file = source_root / "rgb_original_results.txt"

# Write results to text file
with open(output_file, "w") as f:
    f.write("RGB Original Results\n")
    f.write("====================\n\n")
    f.write(f"Total streetlights detected: {total_streetlights}\n")
    f.write(f"ON streetlights: {total_on}\n")
    f.write(f"DIM streetlights: {total_dim}\n")
    f.write(f"OFF streetlights: {total_off}\n")

print(f"Results saved to {output_file}")

**Saving results of rgb analysis per image data on original dataset**

In [ ]:
csv_file = source_root / "rgb_original_per_image.csv"

with open(csv_file, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["image_name", "streetlight_count", "on", "dim", "off"]
    )

    writer.writeheader()

    for result in savith_detailed_results:
        writer.writerow({
            "image_name": result["image_name"],
            "streetlight_count": result["streetlight_count"],
            "on": result["on"],
            "dim": result["dim"],
            "off": result["off"]
        })

print(f"Per-image RGB results saved to: {csv_file}")

**Performing preprocessing on images then doing rgb analysis and storing in csv file**

In [ ]:
model_path = source_root / "runs" / "detect" / "streetlight_savith_best" / "weights" / "best.pt"
model = YOLO(model_path)

# Output CSV file
csv_file = source_root / "rgb_brightened_clahe_per_image.csv"

# Results storage
preprocessed_results = []
total_streetlights = 0
total_on = 0
total_dim = 0
total_off = 0


# Loop through Savith's detected images
for img_path in sorted(savith_images_dir.iterdir()):

    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    image = cv2.imread(str(img_path))

    if image is None:
        print(f"Could not read image: {img_path.name}")
        continue

    # Apply preprocessing: brighten first, then CLAHE
    brightened_image = brighten_image(image)
    preprocessed_image = apply_clahe(brightened_image)

    # Validating using trained YOLO model on preprocessed image
    results = model.predict(
        source=preprocessed_image,
        conf=0.10,
        verbose=False
    )

    boxes = results[0].boxes
    num_lights = len(boxes)

    on_count = 0
    dim_count = 0
    off_count = 0

    per_image_states = []

    for box in boxes:

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        # Keeping coordinates inside image
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(preprocessed_image.shape[1], x2)
        y2 = min(preprocessed_image.shape[0], y2)

        crop = preprocessed_image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        # RGB / brightness classification on preprocessed crop
        state, brightness = classify_streetlight_state(crop)

        if state == "on":
            on_count += 1
            total_on += 1

        elif state == "dim":
            dim_count += 1
            total_dim += 1

        elif state == "off":
            off_count += 1
            total_off += 1

        per_image_states.append({
            "bbox": (x1, y1, x2, y2),
            "state": state,
            "brightness": brightness
        })

        total_streetlights += 1

    preprocessed_results.append({
        "image_name": img_path.name,
        "streetlight_count": num_lights,
        "on": on_count,
        "dim": dim_count,
        "off": off_count,
        "details": per_image_states
    })


# Save per-image results to CSV
with open(csv_file, "w", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=["image_name", "streetlight_count", "on", "dim", "off"]
    )

    writer.writeheader()

    for result in preprocessed_results:
        writer.writerow({
            "image_name": result["image_name"],
            "streetlight_count": result["streetlight_count"],
            "on": result["on"],
            "dim": result["dim"],
            "off": result["off"]
        })

print("Brightened + CLAHE RGB analysis complete.")
print("Total streetlights:", total_streetlights)
print("On:", total_on)
print("Dim:", total_dim)
print("Off:", total_off)
print(f"Per-image results saved to: {csv_file}")

**Comparing results of rgb analysis on original images vs preprocessed images**

In [ ]:
# Load CSV files
original_df = pd.read_csv("rgb_original_per_image.csv")
preprocessed_df = pd.read_csv("rgb_brightened_clahe_per_image.csv")

# Calculating totals
original_on = original_df["on"].sum()
original_dim = original_df["dim"].sum()
original_off = original_df["off"].sum()
original_total = original_df["streetlight_count"].sum()

preprocessed_on = preprocessed_df["on"].sum()
preprocessed_dim = preprocessed_df["dim"].sum()
preprocessed_off = preprocessed_df["off"].sum()
preprocessed_total = preprocessed_df["streetlight_count"].sum()

# Categories
categories = ["ON", "DIM", "OFF", "TOTAL"]

# Values
original_values = [
    original_on,
    original_dim,
    original_off,
    original_total
]

preprocessed_values = [
    preprocessed_on,
    preprocessed_dim,
    preprocessed_off,
    preprocessed_total
]

# Bar positions
x = range(len(categories))
width = 0.35

# Create figure
plt.figure(figsize=(10, 6))

# Original bars
plt.bar(
    [i - width/2 for i in x],
    original_values,
    width=width,
    label="Original RGB"
)

# Preprocessed bars
plt.bar(
    [i + width/2 for i in x],
    preprocessed_values,
    width=width,
    label="Brightened + CLAHE RGB"
)

# Labels
plt.xticks(x, categories)
plt.ylabel("Number of Streetlights")
plt.xlabel("Streetlight State")
plt.title("Comparison of RGB Analysis Results")

for i, v in enumerate(original_values):
    plt.text(i - width/2, v + 2, str(v), ha='center')

for i, v in enumerate(preprocessed_values):
    plt.text(i + width/2, v + 2, str(v), ha='center')

plt.legend()
plt.tight_layout()
plt.show()

### ANALYSIS

The graph compares the RGB based streetlight state classification results between the original images and the images enhanced using brightening and CLAHE preprocessing. The preprocessing significantly change both the total number of deductive streetlights and the distribution of ON, DIM and OFF classifications.
For the ON category , the preprocessed images produced 2084 street lights which were turned on compared to 1744 in the original images without any preprocessing. This shows an increase of 340 detections after preprocessing, suggesting that brightening and CLAHE improved illumination and contrast which made the active streetlights easier to identify. Dimly visible St. lights in the original data set were likely enhanced enough to be classified as fully turned on after preprocessing.

In contrast the dim category decreased sharply from 1192 in the original images down to 297. This large reduction indicates that many streetlights previously considered dim in the original images became clearer and brighter after preprocessing which caused them to shift into the on category. CLAHE improves local contrast, while brightness enhancement increases pixel intensity both of which strengthen the visibility of partially illuminated lights.

The OFF Category also reduced dramatically from 413 to only 38 detections. This suggests that preprocessing helps reveal hidden or poorly illuminated features that were difficult to observe in the original dark images. Many streetlights initially classified as turned off might have been reclassified after enhancement because the preprocessing amplified light regions and improved feature visibility which affected the model identifying the streetlights and reclassifying them.

However, the total number of detected streetlights decreased from 3349 in the original images to 2419 after preprocessing. This indicates that although preprocessing improved visibility and classification quality it may also have caused some reductions to be lost. Excessive brightening or contrast enhancement can alter object boundaries which introduce noise or change image characteristics enough to reduce Yolo detection sensitivity for certain streetlights.

Overall, the preprocessing pipeline appears highly effective for improving the recognition of illuminated streetlights, especially converting dim and of detections into on detections. The results demonstrate that brightening and clahe significantly improved the clarity and visibility of streetlights in low light environments.  However, the reduction in total deduction suggests a trade off between enhanced visibility and object detection consistency. This means that preprocessing improves classification confidence but may reduce overall detection count in some cases.


# LLM Reporting

In [ ]:
!pip install -q openai
!pip install -q -U google-genai
import os
import json
from openai import AsyncOpenAI
from google import genai
from google.genai import types
from dotenv import load_dotenv
import base64
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

**Defining functions for reporting**

In [ ]:
OPENAI_API_KEY = "Your OpenAI API Key Here"

In [ ]:
async def gen_prompt(data):
    
    prompt = f"""
        You are an AI-powered streetlight monitoring assistant.

        An uploaded streetlight image is provided together with ML detection results.
        Use both the image and the detection data to generate a professional, human-readable report.

        ML Detection Results:
        - Total Streetlights: {data["streetlight_count"]}
        - ON Lights: {data["on"]}
        - DIM Lights: {data["dim"]}
        - OFF Lights: {data["off"]}
        - Detection Details: {data["details"]}

        Instructions:
        - Analyze the uploaded image together with the ML output.
        - Describe the overall streetlight condition naturally.
        - Mention operational, dim, and faulty streetlights.
        - Highlight maintenance concerns if necessary.
        - Keep the response concise, professional, and humanized.
        - Do not mention Base64 data in the response.

        Return ONLY in this JSON format:

        {{
        "output": "your human-readable report here"
        }}
    """
    
    return prompt

In [ ]:
async def gpt(prompt, base64_image):
    
    gpt_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    
    chat_prompt = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }
    ]

    completion = await gpt_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=chat_prompt,
        max_tokens=800,
        temperature=0,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )

    response = completion.choices[0].message.content
    return json.loads(response)

In [ ]:
async def llm_reporting(data):
    
    base64_image = data["uploaded_image"]
    prompt = await gen_prompt(data)
    response = await gpt(prompt, base64_image)
    
    return response

Sample image for reporting

In [ ]:
img_path_long = Path("/Users/hamizhassan16/Documents/MOP-Code/datascience/usecases/DEPENDENCIES/UC00229_Smart_Streetlight_Fault_Detection")
img_path = img_path_long / "ExDark" / "Bicycle" / "2015_00002.png"
with open(img_path, "rb") as img_file:
    encoded_image = base64.b64encode(
        img_file.read()
    ).decode("utf-8")

In [ ]:
# Open image
image = Image.open(img_path)

# Display image
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

data = {
    "uploaded_image": encoded_image,
    "streetlight_count": len("boxes"),
    "on": "on_count",
    "dim": "dim_count",
    "off": "off_count",
    "details": "details"
}

response = await llm_reporting(data)
response["output"]

# UI code and walkthrough
**Using Streamlit**

========================================
FILE: backend_api/api.py
========================================

import sqlite3
import json
import os 
from datetime import datetime
from fastapi import FastAPI, UploadFile, File
from typing import List
from fastapi.staticfiles import StaticFiles
from backend_api.models.cv_model import analyse_image
from LLM.llm_reporting import llm_reporting
import base64

#start API app
app = FastAPI()

#store image uploads
upload_directory = "uploads"
os.makedirs(upload_directory, exist_ok = True)

app.mount("/uploads", StaticFiles(directory = "uploads"), name = "uploads")

#database path
db_path = "reports.db"

def init_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    #create new database to store CV model analysis and LLM report 
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            report_id TEXT,
            timestamp TEXT,
            results TEXT
        )
    """)

    conn.commit()
    conn.close()
    
init_db()

@app.get("/")
def root():
    return {"backend": "working"}

#DETECTION ENDPOINT FOR CV
@app.post("/detect")
async def detect_lights(files: List[UploadFile] = File(...)):
    #create unique report folders for each individual report formatted y/m/d (year month day) and h/m/s (hours minutes seconds)
    report_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = os.path.join(upload_directory, report_id)
    os.makedirs(report_path, exist_ok = True)
    
    #for storing in the database and displaying results 
    saved_files = []
    results = []
    
    #process each individual file 
    for file in files: 
        contents = await file.read() 

        file_path = os.path.join(report_path, file.filename)

        #write uploaded image to the disk 
        with open(file_path, "wb") as f: 
            f.write(contents)

        #run CV model analysis
        analysis = analyse_image(file_path)

        #store result for a specific image, image = original file name and analysis = CV model analysis 
        results.append({
            "image": file.filename,
            "analysis": analysis, 
            "uploaded_img": base64.b64encode(contents).decode("utf-8")
        })

        saved_files.append(file.filename)
    
    #return final response 
    return {        
        "report_id": report_id, 
        "results": results
    }
    
#REPORT ENDPOINT FOR LLM 
@app.post("/report")
async def generate_report(data: dict): 
    #get unique report ids 
    report_id = data["report_id"]

    final_results = []

    #loop through each image 
    for item in data["results"]:

        analysis = item["analysis"]

        #send analysis to LLM for report generation 
        llm_result = await llm_reporting({
            "analysis": analysis,
            "uploaded_img": item.get("uploaded_img")
        })

        #store results 
        final_results.append({
            "image": item["image"],
            "analysis": analysis,
            "report": llm_result["output"]
        })
        
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    #store values of analysis 
    cursor.execute("""
        INSERT INTO reports (report_id, timestamp, results)
        VALUES (?, ?, ?)
    """, (
        report_id,
        datetime.now().isoformat(),
        json.dumps(final_results)
    ))

    conn.commit()
    conn.close()
        
    #send response to the client 
    return {
        "report_id": report_id,
        "results": final_results
    }
    
@app.get("/reports")
def get_reports(): 
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    #fetch all reports with the most recent first
    cursor.execute("""
        SELECT report_id, timestamp, results
        FROM reports
        ORDER BY id DESC
    """)
    
    rows = cursor.fetchall()
    conn.close()
    
    #convert rows to JSON format for API 
    reports = []
    for row in rows:
        reports.append({ 
            "report_id": row[0],
            "timestamp": row[1],
            "results": json.loads(row[2])
        })
    
    #return all reports 
    return {"reports": reports}


========================================
FILE: backend_api/models/cv_model.py
========================================

import os
from ultralytics import YOLO
import cv2
import numpy as np
import base64

BASE_DIR = os.path.dirname(__file__)

MODEL_PATH = os.path.join(BASE_DIR, "best.pt")

# load YOLO model ONCE
model = YOLO(MODEL_PATH)

# -----------------------------
# preprocessing functions
# -----------------------------

def brighten_image(image, alpha=1.5, beta=30):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )

    l_clahe = clahe.apply(l)

    merged = cv2.merge((l_clahe, a, b))

    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)


def brighten_then_clahe(image):
    bright = brighten_image(image)
    return apply_clahe(bright)

# temporary substitute if missing
def double_check_preprocess(image):
    return brighten_then_clahe(image)

# -----------------------------
# classification
# -----------------------------

def classify_streetlight_state(
    crop,
    off_threshold=80,
    dim_threshold=160
):

    if crop is None or crop.size == 0:
        return "unknown", 0

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    pixels = gray.flatten()
    pixels = np.sort(pixels)

    top_pixels = pixels[int(0.9 * len(pixels)):]

    mean_brightness = float(np.mean(top_pixels))

    if mean_brightness < off_threshold:
        state = "off"

    elif mean_brightness < dim_threshold:
        state = "dim"

    else:
        state = "on"

    return state, mean_brightness


# -----------------------------
# MAIN ANALYSIS FUNCTION
# -----------------------------

def analyse_image(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return {
            "error": "Could not load image"
        }

    processed_img = double_check_preprocess(image)

    results = model(
        processed_img,
        conf=0.10,
        verbose=False
    )

    boxes = results[0].boxes
    
    if boxes is None or len(boxes) == 0: 
        return {
            "streetlight_count": 0, 
            "on": 0,
            "dim": 0,
            "off": 0,
            "details": []
        }

    on_count = 0 
    dim_count = 0 
    off_count = 0 

    details = []

    for box in boxes:

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(image.shape[1], x2)
        y2 = min(image.shape[0], y2)

        crop = image[y1:y2, x1:x2]

        state, brightness = classify_streetlight_state(crop)

        if state == "on":
            on_count += 1

        elif state == "dim":
            dim_count += 1

        elif state == "off":
            off_count += 1

        details.append({
            "bbox": [x1, y1, x2, y2],
            "state": state,
            "brightness": brightness
        })

    with open(image_path, "rb") as img_file:
        encoded_image = base64.b64encode(
            img_file.read()
        ).decode("utf-8")

    return {
        "uploaded_img": encoded_image,
        "streetlight_count": len(boxes),
        "on": on_count,
        "dim": dim_count,
        "off": off_count,
        "details": details
    }


========================================
FILE: LLM/llm_reporting.py
========================================

import os
import json

from openai import AsyncOpenAI

from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path(__file__).resolve().parent.parent / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


async def gen_prompt(data):

    analysis = data["analysis"]
    
    prompt = f"""
    You are an AI-powered streetlight monitoring assistant.

    An uploaded streetlight image is provided together with ML detection results.

    ML Detection Results:
    - Total Streetlights: {analysis["streetlight_count"]}
    - ON Lights: {analysis["on"]}
    - DIM Lights: {analysis["dim"]}
    - OFF Lights: {analysis["off"]}
    - Detection Details: {analysis["details"]}

    Instructions:
    - Analyze the uploaded image together with the ML output.
    - Describe the overall streetlight condition naturally.
    - Mention operational, dim, and faulty streetlights.
    - Highlight maintenance concerns if necessary.
    - Keep the response concise and professional.
    - Do not mention Base64 data.

    Return ONLY valid JSON:

    {{
        "output": "your report here"
    }}
    """

    return prompt


async def gpt(prompt, base64_image):

    client = AsyncOpenAI(api_key=OPENAI_API_KEY)

    completion = await client.chat.completions.create(
        model="gpt-4.1-mini",

        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],

        max_tokens=800,
        temperature=0
    )

    response = completion.choices[0].message.content

    return json.loads(response)


async def llm_reporting(data):

    base64_image = data["uploaded_img"]

    prompt = await gen_prompt(data)

    response = await gpt(prompt, base64_image)

    return response


========================================
FILE: User_Interface/ui.py
========================================

import streamlit as st 
import requests
import base64

#PAGECONFIG
st.set_page_config(
    page_title = "Smart Streetlight Fault Detection",
    layout = "centered",
    initial_sidebar_state = "expanded"
)

#send images to FastAPI
def backend_detect(files):
    try: 
        url = "http://localhost:8000/detect"
        response = requests.post(url, files = files)
        return response.json()
    except: 
        return {"error": "Backend unavailable"}

#send detection results to FastAPI
def backend_report(detection_data):
    try: 
        url = "http://localhost:8000/report"
        response = requests.post(url, json = detection_data)
        return response.json()
    except: 
        return {"error": "Backend unavailable"}

def backend_get_reports():
    try:
        response = requests.get("http://localhost:8000/reports")
        return response.json()
    except:
        return {"reports": []}

#SIDEBAR
st.sidebar.title("Navigation")
page = st.sidebar.radio("Go to", ["Homepage", "Detection", "Reports", "Report History", "About"])

#HOMEPAGE 
if page == "Homepage": 
    st.title(":blue[Smart Streetlight Fault Detection] :bulb:", text_alignment = "center")
    
    st.markdown("""
                This project utilises a vision-based system to analyse provided nighttime images and detects faulty streetlights, specifically: 
                - Streetlights that are either 
                    - **not functioning** 
                    - **flickering** 
                    - **producing a weak illumination**
                
                Following this, a maintenence alert will be generated. 
                
                Please proceed to the detection page to get started. """)

#DETECTIONPAGE
elif page == "Detection": 
    st.title("Streetlight Analysis")
    
    #columns
    col1, col2 = st.columns([1, 2])
    
    with col1: 
    #file uploader
        uploaded_files = st.file_uploader("Upload image(s)", accept_multiple_files = True, type = ["jpg", "png"])

    with col2: 
    #loop through uploaded files, display preview, and add a button 
        if uploaded_files: 
            st.header("Preview uploaded images")
            for file in uploaded_files: 
                st.image(file, width="stretch")
                st.markdown("---")
            
            #button for analysis 
            if st.button("Analyse image(s)", type = "primary"):
                with st.spinner("Analysing images..."):
                    #prep files 
                    file_data = [
                        ("files", (file.name, file.getvalue(), file.type))
                        for file in uploaded_files
                    ]
                    
                    #send to backend
                    result = backend_detect(file_data)
                    if isinstance(result, dict) and "results" in result:
                        st.session_state["analysis_result"] = result
                        st.success("Analysis complete!")
                    else:
                        st.error("Backend returned invalid response")
                        st.json(result)
        else:
            st.info("No uploaded images.")

    #DIVIDE PAGE 
    st.markdown("---")

    #analysis results section 
    st.header("Analysis Results")
    if "analysis_result" in st.session_state:
        result_data = st.session_state["analysis_result"]

        #check data is in correct format, if yes then loop through each analysed image result 
        if isinstance(result_data, dict) and "results" in result_data:
            for result in result_data["results"]:

                analysis = result["analysis"]
                st.subheader(result["image"])

                #display uploaded image 
                if "uploaded_img" in analysis:
                    image_bytes = base64.b64decode(
                        analysis["uploaded_img"]
                    )
                    st.image(image_bytes)

                #summary for each image 
                st.write(f"Streetlights: {analysis['streetlight_count']}")
                st.write(f"On: {analysis['on']}")
                st.write(f"Dim: {analysis['dim']}")
                st.write(f"Off: {analysis['off']}")

                st.json(analysis["details"])

                st.markdown("---")
    else:
        st.info("No analysis results")

#REPORTSPAGE
elif page == "Reports": 
    st.title("Maintenance Report")
    
    if st.button("Generate report", type = "primary"): 
        if "analysis_result" not in st.session_state: 
            st.warning("No results available. Please run detection analysis first")
        else: 
            with st.spinner("Generating report..."):
                #retrieve previously stored CV detection results 
                detection_data = st.session_state['analysis_result']
                
                #send to backend 
                report = backend_report(detection_data)
                
            if isinstance(report, dict) and "results" in report:
                st.success("Report generated!")

                #display LLM reports for each image 
                for item in report["results"]: 
                    st.subheader(item["image"])
                    st.write(item["report"])
                    st.markdown("---")
            else:
                st.error("Report failed or invalid response from backend")
                st.json(report)
    
#REPORTHISTORY
elif page == "Report History": 
    st.title("Report History")
    
    #fetch all stored reports from database
    data = backend_get_reports() 
    reports = data.get("reports", [])
    
    if not reports: 
        st.warning("No reports found. ")
    else: 
        #loop through all reports 
        for report in reports:
            st.subheader(f"Report: {report['report_id']}")

            #for each image, display the URL, raw CV analysis, and LLM generated report 
            for item in report["results"]:
                image_name = item["image"]

                image_url = f"http://localhost:8000/uploads/{report['report_id']}/{image_name}"

                st.image(image_url, width = "stretch")

                st.json(item["analysis"])

                if "report" in item:
                    st.success(item["report"])
                else:
                    st.warning("No LLM report available")


#ABOUTPAGE
elif page == "About": 
    st.title(":blue[About Us]", text_alignment = "center")
    
    st.markdown("""
            Our project team is composed of the following members: 
            
            **Syed Hamiz Hassan** 
            - Computer Vision Modelling 
            
                
            **Savith Mundukotuwa** 
            - Data Preparation 
            
            
            **Josh Wong** 
            - User Interface and System Integration
            
            
            **Luke Kankannamge Don** 
            - LLM Reporting System 
            
            
            **Rahul Sheoran** 
            - Model Analysis
             
            """)


========================================
FILE: .env.example
========================================

OPENAI_API_KEY=your_api_key_here

# Model Analysis

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# YOLO MODEL OPTIMISATION ANALYSIS
# ==========================================

# Load trained YOLO model
model_path = source_root / "runs" / "detect" / "streetlight_savith_best" / "weights" / "best.pt"
model = YOLO(model_path)

print("YOLO model loaded successfully.")

# ==========================================
# MODEL VALIDATION
# ==========================================

results = model.val(
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.5,
    verbose=False
)

print("\nValidation completed.")

# ==========================================
# OPTIMISATION ANALYSIS
# ==========================================

confidence_values = [0.25, 0.35, 0.45, 0.55]

analysis_results = []

for conf_value in confidence_values:

    print(f"\nRunning analysis with confidence threshold: {conf_value}")

    result = model.val(
        imgsz=640,
        batch=8,
        conf=conf_value,
        iou=0.5,
        verbose=False
    )

    analysis_results.append({
        "Confidence": conf_value,
        "mAP50": float(result.box.map50),
        "mAP50-95": float(result.box.map),
        "Precision": float(result.box.mp),
        "Recall": float(result.box.mr)
    })

# ==========================================
# SAVE RESULTS
# ==========================================

df = pd.DataFrame(analysis_results)

print("\nOptimisation Results:")
print(df)

df.to_csv(source_root / "yolo_analysis_results.csv", index=False)

# ==========================================
# VISUALISATION
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(df["Confidence"], df["mAP50"], marker="o", label="mAP50")
plt.plot(df["Confidence"], df["Precision"], marker="o", label="Precision")
plt.plot(df["Confidence"], df["Recall"], marker="o", label="Recall")

plt.xlabel("Confidence Threshold")
plt.ylabel("Performance Score")
plt.title("YOLO Model Optimisation Analysis")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(source_root / "yolo_analysis_graph.png")

plt.show()

print("\nAnalysis completed successfully.")
print("Results saved as yolo_analysis_results.csv")
print("Graph saved as yolo_analysis_graph.png")